https://youtu.be/D71n1wKrmNQ?si=KwVsgPu1IRWNGGIY

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
schema = StructType([
    StructField("player", StringType(), True),
    StructField("runs", IntegerType(), True),
    StructField("50s/100s", StringType(), True)
])

data = [("Sachin-IND", 18694, "93/49"), ("Ricky-AUS", 11274, "66/31"),("Lara-WI", 10222, "45/21"),("Rahul-IND", 10355, "95/11"),("Jhonty-SA", 7051, "43/5"),("Hayden-AUS", 8722, "67/19")]
players_df = spark.createDataFrame(data, schema)
display(players_df)

data1 = [("IND", "India"), ("AUS", "Australia"), ("WI", "WestIndies"), ("SA", "SouthAfrica")]
countries_df = spark.createDataFrame(data1,["SRT","country"])
display(countries_df)

player,runs,50s/100s
Sachin-IND,18694,93/49
Ricky-AUS,11274,66/31
Lara-WI,10222,45/21
Rahul-IND,10355,95/11
Jhonty-SA,7051,43/5
Hayden-AUS,8722,67/19


SRT,country
IND,India
AUS,Australia
WI,WestIndies
SA,SouthAfrica


In [0]:
players_df1 = players_df.withColumn("player1",split(col("player"),"-")[0]).withColumn("SRT",split(col("player"),"-")[1]).withColumn("50",split(col("50s/100s"),"/")[0].cast("int")).withColumn("100",split(col("50s/100s"),"/")[1].cast("int"))
display(players_df1)

player,runs,50s/100s,player1,SRT,50,100
Sachin-IND,18694,93/49,Sachin,IND,93,49
Ricky-AUS,11274,66/31,Ricky,AUS,66,31
Lara-WI,10222,45/21,Lara,WI,45,21
Rahul-IND,10355,95/11,Rahul,IND,95,11
Jhonty-SA,7051,43/5,Jhonty,SA,43,5
Hayden-AUS,8722,67/19,Hayden,AUS,67,19


In [0]:
df = players_df1.join(broadcast(countries_df),players_df1.SRT == countries_df.SRT, "left")
display(df)

player,runs,50s/100s,player1,SRT,50,100,SRT,country
Sachin-IND,18694,93/49,Sachin,IND,93,49,IND,India
Ricky-AUS,11274,66/31,Ricky,AUS,66,31,AUS,Australia
Lara-WI,10222,45/21,Lara,WI,45,21,WI,WestIndies
Rahul-IND,10355,95/11,Rahul,IND,95,11,IND,India
Jhonty-SA,7051,43/5,Jhonty,SA,43,5,SA,SouthAfrica
Hayden-AUS,8722,67/19,Hayden,AUS,67,19,AUS,Australia


In [0]:
df = df.select("player1","country","runs","50","100")
display(df)

player1,country,runs,50,100
Sachin,India,18694,93,49
Ricky,Australia,11274,66,31
Lara,WestIndies,10222,45,21
Rahul,India,10355,95,11
Jhonty,SouthAfrica,7051,43,5
Hayden,Australia,8722,67,19


In [0]:
df1 = df.withColumn("sum", col("50") + col("100")).drop("50","100").filter("sum > 90").orderBy(desc("sum")).withColumnRenamed("player1","playername")
display(df1)

playername,country,runs,sum
Sachin,India,18694,142
Rahul,India,10355,106
Ricky,Australia,11274,97
